<table>
  <tr>
    <td><div align="left"><font size="30">Camera and image motion</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200" height="158" style="height:auto"></td>
  </tr>
</table>

(c) Peter Corke 2024

Robotics, Vision & Control: Python, see Chapter 15

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`). If you're running locally you need to have these packages installed.

<div class="alert alert-info">
If you're running on CoLab or in the browser we have to first install machinevisiontoolbox, which is not preinstalled, so the next cell will be slow.
</div>

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'ipympl', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
import numpy as np
from machinevisiontoolbox import CentralCamera
from spatialmath import SE3
import scipy.linalg as lin

%matplotlib inline
import matplotlib.pyplot as plt
np.set_printoptions(linewidth=120, formatter={'float': lambda x: f"{x:8.4g}" if abs(x) > 1e-10 else f"{0:8.4g}"})


***
# Relationship between camera and image-plane motion
Instantiate a projective camera object, centred at the origin and viewing parallel to the world frame z-axis

In [ ]:
camera = CentralCamera.Default()
print(camera)

Define a point in the world

In [ ]:
P = [1, 1, 5]

We will project it to the image plane

In [ ]:
p0 = camera.project_point(P)
p0

Now if we displace the camera slightly (distance `d`) in the x-direction the pixel coordinates will become

In [ ]:
d = 0.1
px = camera.project_point(P, pose=SE3([d, 0, 0]))
px

Using the camera coordinate conventions of RVC2 Fig. 11.5, the camera has moved to the right so the image point has moved to the left. The sensitivity of image motion to camera motion is

In [ ]:
( px - p0) / d

which is an approximation to the partial derivative $\partial p / \partial x$. It shows that 1 m of camera motion would lead to −160 pixel of feature motion in the u-direction. 

We can repeat this for z-axis translation, ie. moving the camera forward


In [ ]:
( camera.project_point(P, pose=SE3([0, 0, d])) - p0) / d

which is an approximation to the partial derivative $\partial p / \partial z$, and  shows equal motion in the u- and v-directions

We can also rotate the camera about the x-axis, equivalent to pitching the camera upwards

In [ ]:
px = camera.project_point(P, pose=SE3.Rx(d))
( px - p0) / d

which is predominantly motion down in the image, remember the v-axis points down in the image. the image motion is predominantly in the v-direction. It is clear that camera motion along and about the different degrees of freedom in SE(3) causes quite different motion of image points.

## Things to try on your own
1. choose a point that is further from the camera and look at how it is projected to the image plane
2. use a smaller value of `d` when computing the numerical approximation to the Jacobian

# Image Jacobian

Consider the camera as a function 
\begin{equation}
p = {\cal P}(P, \mathbf{K}, \xi)
\end{equation}
where $p \in \mathbb{R}^2$ is the image-plane point, $P \in \mathbb{R}^3$ is the world point, $\mathbf{K} \in \mathbb{R}^{3 \times 3}$ is the camera intrinsics and $\xi \in SE(3)$ is the camera pose. Its derivative with respect to time is
\begin{equation}
\dot{p} = \mathbf{J}_p(P, \mathbf{K}, \xi) \nu
\end{equation}
where $\nu = (v_x, v_y, v_z, \omega_x, \omega_y, \omega_z) \in \mathbb{R}^6$ is the velocity of the camera -- the spatial velocity.

$\mathbf{J}_p$ is a Jacobian-like object, but because we have taken the derivative with respect to a pose $\xi \in SE(3)$ rather than a vector it is technically called an _interaction matrix_. However in the visual servoing world it is more commonly called an _image Jacobian_ or a _feature sensitivity matrix_.

The Toolbox CentralCamera class provides the method visjac_p to compute the image Jacobian and for the example above it is


In [ ]:
J = camera.visjac_p( (672, 672), 5)
J

Each column gives the image-plane velocity for the corresponding component of camera velocity, ie. the first column corresponds to camera x-axis motion, the second column corresponds to camera x-axis motion and the last column correpsonds to camera z-axis rotation.  In Out[6] we computed a numerical approximation to the first column,  Out[7] an approximation to the third column, and Out[8] is an approximation to the fourth column.

If the camera is moving at 1m/s in the x-axis direction the velocity at the pixel will be

In [ ]:
J @ [1, 0, 0, 0, 0, 0]

in units of pixel/s.

If the velocity was 1rad/s around the z-axis the velocity at the pixel will be

In [ ]:
J @ [0, 0, 0, 0, 0, 1]

and if the camera velocity was 1m/s in the x-axis direction *and* 1rad/s in around the z-axis the velocity at the pixel will be

In [ ]:
J @ [1, 0, 0, 0, 0, 1]

# Optical flow fields

We just computed the image Jacobian for a single point close to the centre of the image plane.  Imagine now that we compute it for a grid over the image plane and then see how the image-plane velocity at each point varies with camera veloicty.

The example below provides you with a slider for each component of camera velocity.  As you adjust them the corresponding optical flow field is displayed. Have a play and try to understand what this is showing you.  What happens if you motion in several dimensions of the camera velocity space?

In [ ]:
import ipywidgets as widgets

camera = CentralCamera.Default() # create camera object


@widgets.interact
def animate( x =  widgets.FloatSlider(value=0, description='X:',  min=-1, max=1),
             y =  widgets.FloatSlider(value=0,   description='Y:',  min=-1, max=1),
             z =  widgets.FloatSlider(value=0,   description='Z:',  min=-1, max=1),
             rx = widgets.FloatSlider(value=0,   description='rX:', min=-1, max=1),
             ry = widgets.FloatSlider(value=0,   description='rY:', min=-1, max=1),
             rz = widgets.FloatSlider(value=0,   description='rZ:', min=-1, max=1) ):

    Z = 2  # distance to the grid of points
    vel = [x, y, z, rx, ry, rz]  # camera velocity from sliders, as a column vector
    
    # setup the plot window, fixed scale, unity aspect ratio, grid lines on
    plt.figure(figsize=(8,8))
    ax = plt.gca()
    plt.grid(True)
    ax.set_aspect('equal')
    ax.set_facecolor('yellow')
    plt.xlabel('u (pixels)')
    plt.ylabel('v (pixels)')
    plt.title('Velocity on camera image plane')
    plt.xlim(0, 1000)
    ax.set_ylim(1000, 0)  # inverted y-axis


    # compute optical flow over a grid of points
    a = np.arange(-1000, 1000, 50)  # set of flow points

    [U, V] = np.meshgrid(a, a, indexing='ij')
    du = np.empty(U.shape)
    dv = np.empty(U.shape)
    
    for i in range(U.shape[1]):
        for j in range(U.shape[0]):
            pdot = camera.visjac_p( ([U[i,j], V[i,j]]), Z) @ vel
            du[i,j] = pdot[0]
            dv[i,j] = pdot[1]

    # plot the flow vectors
    #  -dv is to overcome a bug with quiver and flipped vertical axis
    plt.quiver(U, V, du, -dv, scale=8000)
    
    

## Things to try on your own
1. Change the camera to a 4mm focal length, and see the change in the shape of the optical flow fields for rotation about the x- and y-axes.

# Imperceptible motion
Define a point in the world and compute its image Jacobian

In [ ]:
P = [0, 0, 5]
p = camera.project_point(P)
J = camera.visjac_p( p, 5)
J

The Jacobian is $2 \times 6$ which means it has a nullity of 4, that is, there are four possible velocities that will cause zero image plane motion, as will any linear combination of these velocities

In [ ]:
N = lin.null_space(J)
N.shape

We can quickly verify this is the case for the first column which is a camera velocity of

In [ ]:
N[:,0]

and the resulting image-plane velocity is

In [ ]:
J @ N[:,0]

which is effectively zero.

# Ambiguous motion

In [ ]:
P = [1, 1, 5]
p = camera.project_point(P)
J = camera.visjac_p( p, 5)
J

In [ ]:
np.set_printoptions(precision=3, floatmode='maxprec')
np.set_printoptions(suppress=True)
@widgets.interact
def animate( f =  widgets.FloatSlider(value=8, description='f (mm): ',  min=1, max=100) ):
    camera = CentralCamera.Default(f=f*1e-3) # create camera object
    p = camera.project_point(P)
    J = camera.visjac_p( p, 5)
    print(J)